# CE224: transfer learning, kiểm tra theo quả và export NCNN

Giữ YOLO11n pretrained và đầu vào 640 cho Pi 5. Không tăng epoch/model chỉ vì
validation gần 100%: lần chạy cũ đạt val mAP50–95 99,23%, test 93,16%; fine-tune
chỉ hơn warmup 0,14 điểm phần trăm. Lần chạy cũ đã được sao lưu trước khi sửa.

Thay đổi chính:
- So sánh baseline và recipe giữ màu trên cùng 3 fold, nhóm theo quả, chỉ dùng
  train+validation. Mọi frame/biến thể cùng quả ở chung một phía của mỗi fold.
- Ảnh augmentation offline chỉ được dùng cho train của fold, không đưa vào val.
- Recipe giữ màu dùng augmentations=[] để tắt cả ToGray/Blur/CLAHE mặc định;
  giữ augmentation hình học và sáng nhẹ của YOLO. Không giả định đây chắc chắn
  là nguyên nhân lỗi; chọn recipe theo trung bình mAP của 3 fold, báo cả độ lệch.
- Báo confusion matrix ở confidence vận hành 0.4, lỗi hư → chín theo từng quả.
- Export NCNN, so sánh với .pt trên cùng validation/CPU và đóng gói ZIP.

Test đã được xem để phân tích lỗi ở lần trước, nên kết quả mới trên test cũ là
benchmark tham khảo, không phải đánh giá blind hoàn toàn. Không chuyển quả test
vào train hoặc chọn recipe bằng test. Không tự đổi nhãn/đánh dấu vàng-hư bằng màu.

Tham khảo: [Ultralytics augmentation](https://docs.ultralytics.com/guides/yolo-data-augmentation/),
[cross-validation](https://docs.ultralytics.com/guides/kfold-cross-validation/),
[NCNN](https://docs.ultralytics.com/integrations/ncnn/).


In [ ]:
# Giữ phiên bản Ultralytics đã dùng; không cài lại PyTorch/CUDA Kaggle.
%pip install ultralytics==8.4.194 albumentations==2.0.8 PyYAML Pillow


## 1. Cách chạy Kaggle

Add Input dataset YOLO đã label, giữ data.yaml/train/valid/test. Bật GPU và
Internet. Chạy audit/preview trước rồi đặt RUN_TRAINING=True.
RUN_CV=True chạy 3 fold × 2 recipe, mỗi lượt gồm warmup và fine-tune; thời gian
dài hơn đáng kể một lần train. Hai recipe dùng cùng seed, fold và lịch học.
Sau CV, notebook train mới từ COCO bằng recipe được chọn, trên split gốc.

Nếu chỉ muốn export checkpoint đã train: giữ RUN_TRAINING=False và điền
EXISTING_CHECKPOINT bằng đường dẫn best.pt đã upload; không cần train lại.
RUN_EXPORT_NCNN=True sẽ export, validate và tạo ZIP. Kaggle cần Internet để tải
pretrained/export dependencies. Kết quả ghi trong /kaggle/working/ce224_training.
Không sửa ảnh/nhãn input. RUN_FINAL_TEST mặc định False.

Không biết nhóm vàng-hư cụ thể: notebook chưa tự tăng trọng số theo ngoại hình.
Xem các báo cáo operating.json và ảnh mistakes trước khi quyết định bổ sung mẫu.


In [ ]:
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
import csv
import hashlib
import json
import math
import os
import random
import re
import shutil
import statistics

import yaml
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

IS_KAGGLE = Path("/kaggle/input").is_dir() and Path("/kaggle/working").is_dir()
DATASET_OVERRIDE = None  # Nếu cần: "/kaggle/input/.../CE224.v1-YOLOv8"

def find_dataset():
    if DATASET_OVERRIDE is not None:
        root = Path(DATASET_OVERRIDE).resolve()
        if not (root / "data.yaml").is_file():
            raise FileNotFoundError(f"DATASET_OVERRIDE không chứa data.yaml: {root}")
        return root
    if IS_KAGGLE:
        candidates = sorted({path.parent.resolve() for path in Path("/kaggle/input").rglob("data.yaml")
                             if all((path.parent / split / "images").is_dir()
                                    for split in ("train", "valid", "test"))})
        if len(candidates) == 1:
            return candidates[0]
        if not candidates:
            raise FileNotFoundError("Add Input dataset đã giải nén, có data.yaml và train/valid/test/images")
        raise ValueError("Có nhiều dataset. Điền DATASET_OVERRIDE bằng một đường dẫn: "
                         + ", ".join(str(path) for path in candidates))
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        candidate = directory / "dataset" / "CE224.v1-YOLOv8"
        if (candidate / "data.yaml").is_file():
            return candidate
    raise FileNotFoundError("Sửa DATASET_OVERRIDE: cần thư mục chứa data.yaml, train/, valid/, test/")

DATASET_ROOT = find_dataset()
OUTPUT_ROOT = Path("/kaggle/working/ce224_training") if IS_KAGGLE else DATASET_ROOT.parent / "training_runs"
os.environ.setdefault("YOLO_CONFIG_DIR", str(OUTPUT_ROOT / ".ultralytics"))

import numpy as np
import torch
import ultralytics
from ultralytics import YOLO

SEED = 42
DEVICE = 0 if torch.cuda.is_available() else "cpu"
BATCH = 16 if torch.cuda.is_available() else 4
IMGSZ = 640
PRETRAINED = "yolo11n.pt"
WARMUP_EPOCHS = 15
FINETUNE_EPOCHS = 60
DEPLOY_CONF = 0.4
MATCH_IOU = 0.5  # Ghép detection với ground truth, khác NMS iou=0.7.
RUN_TRAINING = False
RUN_CV = True
N_FOLDS = 3
RUN_FINAL_TEST = False
RUN_EXPORT_NCNN = True
EXISTING_CHECKPOINT = None  # Ví dụ: "/kaggle/input/.../best.pt"; không bật train cùng lúc.
NCNN_MAX_MAP_DROP = 0.005  # Cờ kiểm tra: mAP giảm tối đa 0,5 điểm phần trăm.

# baseline có Albumentations mặc định; preserve_color tắt bộ mặc định đó.
RECIPES = {"preserve_color": [], "baseline_default": None}
SELECTED_RECIPE = "preserve_color"  # Dùng khi RUN_CV=False; chưa được chứng minh tốt hơn.

print("Dataset:", DATASET_ROOT, "| Output:", OUTPUT_ROOT)
print("Ultralytics:", ultralytics.__version__, "| Torch:", torch.__version__)
print("Device:", DEVICE, "| batch:", BATCH)
if DEVICE == "cpu":
    print("Kaggle: bật GPU trước khi train. Export NCNN dùng CPU.")


## 2. Audit dữ liệu trước khi train

Kiểm tra ảnh đọc được, file nhãn tương ứng, 5 trường mỗi box, class ID, giá trị
hữu hạn, box nằm trong ảnh, box trùng, ảnh nền, tên nhóm và split. Fail sớm nếu
có lỗi cấu trúc hoặc leakage. Tên thư mục `valid` được hiểu là validation.
Không sửa data.yaml gốc: đường dẫn `../train/images` trong export sẽ được chuẩn
hóa vào file YAML riêng trong output của lần train.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_NAMES = {"damaged", "green", "riped"}

def audit_dataset(root):
    config = yaml.safe_load((root / "data.yaml").read_text(encoding="utf-8"))
    raw_names = config.get("names")
    if isinstance(raw_names, dict):
        raw_names = {int(key): value for key, value in raw_names.items()}
        if set(raw_names) != set(range(len(raw_names))):
            raise ValueError("Class IDs phải liên tiếp từ 0")
        names = [raw_names[index] for index in range(len(raw_names))]
    elif isinstance(raw_names, list):
        names = raw_names
    else:
        raise ValueError("data.yaml thiếu names hợp lệ")
    if len(names) != 3 or set(names) != EXPECTED_NAMES or config.get("nc", 3) != 3:
        raise ValueError(f"Cần đúng damaged/green/riped; nhận {names}")

    records, errors, report = [], [], {}
    group_splits, hash_splits = defaultdict(set), defaultdict(set)
    for split in ("train", "valid", "test"):
        image_directory, label_directory = root / split / "images", root / split / "labels"
        images = sorted(p for p in image_directory.glob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
        if not images or not label_directory.is_dir():
            raise ValueError(f"Thiếu images/labels hoặc tập rỗng: {split}")
        stems = [p.stem for p in images]
        if len(stems) != len(set(stems)):
            errors.append(f"{split}: nhiều ảnh chung stem, gây nhầm file nhãn")
        orphan = [p.name for p in label_directory.glob("*.txt") if p.stem not in set(stems)]
        if orphan:
            errors.append(f"{split}: nhãn không có ảnh: {orphan[:3]}")
        instances, image_counts, dimensions, empty = Counter(), Counter(), Counter(), 0
        split_groups = defaultdict(set)
        for path in images:
            try:
                with Image.open(path) as image:
                    image.load()
                    dimensions[str(image.size)] += 1
            except Exception as exc:
                errors.append(f"Ảnh hỏng {path.name}: {exc}")
                continue
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
            hash_splits[digest].add(split)
            label_path = label_directory / f"{path.stem}.txt"
            if not label_path.is_file():
                errors.append(f"Thiếu nhãn (kể cả ảnh nền phải có file rỗng): {path.name}")
                continue
            boxes, rows_seen = [], set()
            for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), 1):
                if not line.strip():
                    continue
                try:
                    fields = line.split()
                    if len(fields) != 5:
                        raise ValueError("cần class_id x_center y_center width height")
                    class_id = int(fields[0])
                    x, y, width, height = map(float, fields[1:])
                    if not 0 <= class_id < len(names):
                        raise ValueError("class_id ngoài names")
                    if not all(math.isfinite(value) for value in (x, y, width, height)):
                        raise ValueError("NaN/inf")
                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < width <= 1 and 0 < height <= 1):
                        raise ValueError("tọa độ/kích thước ngoài 0..1")
                    if min(x - width/2, y - height/2) < -1e-5 or max(x + width/2, y + height/2) > 1 + 1e-5:
                        raise ValueError("box vượt ảnh")
                    box = (class_id, x, y, width, height)
                    if box in rows_seen:
                        raise ValueError("box trùng trong cùng ảnh")
                    rows_seen.add(box)
                    boxes.append(box)
                    instances[names[class_id]] += 1
                except (ValueError, TypeError) as exc:
                    errors.append(f"{label_path.name}:{line_number}: {exc}")
            for class_id in {box[0] for box in boxes}:
                image_counts[names[class_id]] += 1
            if not boxes:
                empty += 1
            match = re.match(r"^(damaged|green|riped)__(fruit\d+)__", path.stem)
            if match:
                label, fruit = match.groups()
                group_splits[label, fruit].add(split)
                split_groups[label].add(fruit)
                if not boxes or any(names[box[0]] != label for box in boxes):
                    errors.append(f"Lớp nhãn khác tên nguồn hoặc thiếu box: {path.name}")
            elif path.stem.startswith("background__"):
                if boxes or split != "train":
                    errors.append(f"Ảnh nền phải rỗng và thuộc train: {path.name}")
            else:
                errors.append(f"Không nhận dạng nhóm quả từ tên file: {path.name}")
            if split != "train" and "__aug" in path.stem:
                errors.append(f"Augmentation ngoài train: {path.name}")
            records.append({"split": split, "image": path, "label": label_path, "boxes": boxes})
        missing_classes = EXPECTED_NAMES - set(instances)
        if missing_classes:
            errors.append(f"{split}: thiếu lớp {sorted(missing_classes)}")
        report[split] = {"images": len(images), "background": empty,
                         "instances": dict(instances), "images_per_class": dict(image_counts),
                         "groups": {name: sorted(value) for name, value in split_groups.items()},
                         "dimensions": dict(dimensions)}
    leaked_groups = {str(key): sorted(value) for key, value in group_splits.items() if len(value) > 1}
    leaked_hashes = sum(len(value) > 1 for value in hash_splits.values())
    if leaked_groups or leaked_hashes:
        errors.append(f"Leakage: groups={leaked_groups}, exact_duplicate_hashes={leaked_hashes}")
    if errors:
        raise ValueError("Audit thất bại:\n" + "\n".join(errors[:25]))
    return names, records, report

CLASS_NAMES, RECORDS, AUDIT = audit_dataset(DATASET_ROOT)
print("Class ID mapping:", dict(enumerate(CLASS_NAMES)))
print(json.dumps(AUDIT, ensure_ascii=False, indent=2))
print("PASS: cấu trúc nhãn, ảnh nền và nhóm train/valid/test.")
print("Lưu ý: val/test không có ảnh nền độc lập, chưa đo đầy đủ false positive trên băng tải trống.")


In [ ]:
# Đối chiếu manifest nếu vẫn có bộ đã chuẩn bị bên cạnh export; không phải điều kiện để dùng Colab.
manifest_path = DATASET_ROOT.parent / "ce224_augmented" / "manifest.csv"
if manifest_path.is_file():
    with manifest_path.open(encoding="utf-8", newline="") as file:
        expected = {Path(row["output"]).stem: row for row in csv.DictReader(file)}
    observed, mismatches = set(), []
    for record in RECORDS:
        stem = record["image"].stem.split(".rf.")[0]
        original_stem = re.sub(r"_(jpg|jpeg|png)$", "", stem)
        row = expected.get(original_stem)
        if row is None:
            mismatches.append(f"Không tìm thấy trong manifest: {record['image'].name}")
            continue
        observed.add(original_stem)
        expected_split = "valid" if row["split"] == "val" else row["split"]
        if record["split"] != expected_split:
            mismatches.append(f"Sai split: {record['image'].name}")
        if record["boxes"] and any(CLASS_NAMES[box[0]] != row["class"] for box in record["boxes"]):
            mismatches.append(f"Sai class theo manifest: {record['image'].name}")
    if mismatches:
        raise ValueError("\n".join(mismatches[:20]))
    omitted = Counter((row["split"], row["class"]) for key, row in expected.items() if key not in observed)
    print("Ảnh trong manifest không có trong export:", dict(omitted))
    print("Tổng ảnh thiếu:", sum(omitted.values()), "| Các ảnh có mặt khớp lớp/split.")
else:
    print("Không có manifest cạnh dataset; đã kiểm tra leakage qua tên nhóm và hash ảnh.")


## 3. Xem nhãn trước khi train

Preview train/validation với seed cố định. Box phải bao toàn quả, không chỉ vết hư.
Nếu ảnh mờ, box sai hoặc lớp sai, sửa trong công cụ annotation và export lại;
không sửa tự động nhãn hoặc chuyển ảnh giữa các tập ở notebook này.


In [ ]:
colors = {"damaged": "#ff6978", "green": "#39d98a", "riped": "#ffcb47"}
preview_rows = []
rng = random.Random(SEED)
for split in ("train", "valid"):
    for name in CLASS_NAMES:
        candidates = [record for record in RECORDS if record["split"] == split
                      and any(CLASS_NAMES[box[0]] == name for box in record["boxes"])]
        preview_rows.append(rng.choice(candidates))
canvas = Image.new("RGB", (960, 660), "#15202b")
for index, record in enumerate(preview_rows):
    with Image.open(record["image"]) as source:
        image = source.convert("RGB")
    draw = ImageDraw.Draw(image)
    for class_id, x, y, width, height in record["boxes"]:
        box = ((x-width/2)*image.width, (y-height/2)*image.height,
               (x+width/2)*image.width, (y+height/2)*image.height)
        draw.rectangle(box, outline=colors[CLASS_NAMES[class_id]], width=3)
        draw.text((box[0], max(0, box[1]-14)), CLASS_NAMES[class_id], fill=colors[CLASS_NAMES[class_id]])
    x_offset, y_offset = (index % 3)*320, (index // 3)*330
    canvas.paste(ImageOps.contain(image, (316, 302)), (x_offset, y_offset+24))
    ImageDraw.Draw(canvas).text((x_offset+5, y_offset+5), f"{record['split']} / {record['image'].name[:28]}", fill="white")
display(canvas)


## 4. Chuẩn bị fold theo quả

Nguồn CV chỉ gồm train+valid. Phân nhóm bằng nhãn + fruitN, không chia từng ảnh.
Mỗi lớp được phân phối nhóm qua 3 fold; 6 quả chín development thành 2 quả val
mỗi fold. Ảnh nền cùng phiên chưa xác định luôn nằm train, không tạo bằng chứng
đánh giá false positive trên nền độc lập. Ảnh __aug chỉ ở train của mỗi fold.

CV là phép đánh giá, không tạo thêm quả thật. Final train vẫn giữ split gốc để
có validation cho early stopping; không chọn fold có điểm cao nhất làm model cuối.


In [ ]:
def fruit_group(record):
    match = re.match(r"^(damaged|green|riped)__(fruit\d+)__", record["image"].stem)
    return f"{match[1]}:{match[2]}" if match else "background"


def make_group_folds(records, count=3):
    development = [r for r in records if r["split"] in ("train", "valid")]
    by_class = defaultdict(set)
    for record in development:
        group = fruit_group(record)
        if group != "background":
            by_class[group.split(":")[0]].add(group)
    assignment = {}
    rng = random.Random(SEED)
    for label in CLASS_NAMES:
        groups = sorted(by_class[label])
        if len(groups) < count:
            raise ValueError(f"Không đủ {count} nhóm cho {label}")
        rng.shuffle(groups)
        assignment.update({group: index % count for index, group in enumerate(groups)})
    folds = []
    for fold in range(count):
        train, valid = [], []
        for record in development:
            group = fruit_group(record)
            if group == "background" or assignment[group] != fold:
                train.append(record)
            elif "__aug" not in record["image"].stem:
                valid.append(record)
        train_groups = {fruit_group(r) for r in train} - {"background"}
        valid_groups = {fruit_group(r) for r in valid}
        if train_groups & valid_groups or any(r["split"] == "test" for r in train + valid):
            raise ValueError("Leakage trong CV")
        folds.append({"train": train, "valid": valid})
    return folds


def stage_dataset(split_records, directory):
    directory.mkdir(parents=True, exist_ok=False)
    manifest = []
    for split, records in split_records.items():
        for kind in ("images", "labels"):
            (directory / split / kind).mkdir(parents=True)
        for record in records:
            shutil.copy2(record["image"], directory / split / "images" / record["image"].name)
            shutil.copy2(record["label"], directory / split / "labels" / record["label"].name)
            manifest.append({"source": str(record["image"]), "source_split": record["split"],
                             "split": split, "group": fruit_group(record)})
    config = {"path": directory.as_posix(), "train": "train/images", "val": "valid/images",
              "names": CLASS_NAMES, "nc": len(CLASS_NAMES)}
    if "test" in split_records:
        config["test"] = "test/images"
    data_yaml = directory.parent / "data.yaml"
    data_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
    (directory.parent / "split_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return data_yaml


FOLDS = make_group_folds(RECORDS, N_FOLDS)
for index, fold in enumerate(FOLDS):
    print("Fold", index + 1, "| images:", {key: len(value) for key, value in fold.items()},
          "| val fruits:", dict(Counter(group.split(":")[0] for group in {fruit_group(r) for r in fold["valid"]})))


## 5. Đánh giá ở ngưỡng vận hành

Predict ở conf=0.4 rồi ghép box một-một với IoU=0.5, tính confusion matrix thực
tại ngưỡng này. Không lấy bảng P/R tối ưu F1 làm kết quả ở confidence 0.4.
Hàng = dự đoán, cột = nhãn thật; cột cuối là FP, hàng cuối là FN.
Lưu cả số lỗi theo quả, danh sách ảnh lỗi và tối đa 12 preview để xem.


In [ ]:
from ultralytics.utils.metrics import ConfusionMatrix


def operating_evaluation(checkpoint, records, directory, device=DEVICE):
    directory.mkdir(parents=True, exist_ok=False)
    model = YOLO(str(checkpoint), task="detect")
    matrix = ConfusionMatrix(names=dict(enumerate(CLASS_NAMES)))
    by_fruit = defaultdict(lambda: {"frames": 0, "correct": 0, "damaged_to_riped": 0, "missed": 0})
    errors = []
    damaged_id, ripe_id = CLASS_NAMES.index("damaged"), CLASS_NAMES.index("riped")
    predictions = model.predict(source=[str(r["image"]) for r in records], stream=True,
                                imgsz=IMGSZ, rect=False, conf=DEPLOY_CONF, iou=0.7,
                                batch=1, device=device, save=False, verbose=False)
    seen = 0
    for record, result in zip(records, predictions, strict=True):
        seen += 1
        if Path(result.path).resolve() != record["image"].resolve():
            raise RuntimeError("Thứ tự prediction khác nguồn")
        height, width = result.orig_shape
        boxes = torch.tensor(record["boxes"], dtype=torch.float32).reshape(-1, 5)
        gt_xyxy = torch.empty((len(boxes), 4), dtype=torch.float32)
        if len(boxes):
            gt_xyxy[:, 0] = (boxes[:, 1] - boxes[:, 3]/2)*width
            gt_xyxy[:, 1] = (boxes[:, 2] - boxes[:, 4]/2)*height
            gt_xyxy[:, 2] = (boxes[:, 1] + boxes[:, 3]/2)*width
            gt_xyxy[:, 3] = (boxes[:, 2] + boxes[:, 4]/2)*height
        detections = {"bboxes": result.boxes.xyxy.cpu(), "cls": result.boxes.cls.cpu(),
                      "conf": result.boxes.conf.cpu()}
        before = matrix.matrix.copy()
        matrix.process_batch(detections, {"bboxes": gt_xyxy, "cls": boxes[:, 0]},
                             conf=DEPLOY_CONF, iou_thres=MATCH_IOU)
        difference = matrix.matrix - before
        group = fruit_group(record)
        by_fruit[group]["frames"] += 1
        by_fruit[group]["correct"] += int(np.trace(difference[:3, :3]))
        by_fruit[group]["damaged_to_riped"] += int(difference[ripe_id, damaged_id])
        by_fruit[group]["missed"] += int(difference[3, :3].sum())
        if difference.sum() > np.trace(difference[:3, :3]):
            errors.append({"image": record["image"].name, "group": group,
                           "true": [CLASS_NAMES[int(b[0])] for b in record["boxes"]],
                           "predicted": [CLASS_NAMES[int(c)] for c in result.boxes.cls.cpu().tolist()],
                           "confidences": result.boxes.conf.cpu().tolist()})
            if len(errors) <= 12:
                result.save(filename=str(directory / f"mistake_{len(errors):02d}.jpg"))
    if seen != len(records):
        raise RuntimeError("Thiếu prediction")
    values = matrix.matrix
    classes = []
    for index, name in enumerate(CLASS_NAMES):
        tp = values[index, index]
        predicted, actual = values[index, :].sum(), values[:, index].sum()
        classes.append({"class": name, "precision": float(tp/predicted) if predicted else 0.0,
                        "recall": float(tp/actual) if actual else 0.0})
    damaged_total = values[:, damaged_id].sum()
    report = {"confidence": DEPLOY_CONF, "match_iou": MATCH_IOU,
              "matrix_order": CLASS_NAMES + ["background"], "matrix": values.astype(int).tolist(),
              "classes": classes, "damaged_to_riped": int(values[ripe_id, damaged_id]),
              "damaged_to_riped_rate": float(values[ripe_id, damaged_id]/damaged_total) if damaged_total else None,
              "by_fruit": dict(by_fruit), "errors": errors}
    (directory / "operating.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    matrix.plot(normalize=False, save_dir=directory)
    matrix.plot(normalize=True, save_dir=directory)
    return report


## 6. Train hai giai đoạn và so sánh recipe

Giữ 15 epoch freeze=10, rồi tối đa 60 epoch freeze=0, LR lần lượt 1e-3/1e-4;
early stopping vẫn hoạt động. Mỗi fold bắt đầu từ COCO, không dùng checkpoint
của fold trước. Hai recipe chỉ khác Albumentations, không đổi nhiều yếu tố một lúc.
Toàn bộ đánh giá dùng rect=False, imgsz=640 để đồng nhất với NCNN fixed input.
CV báo mean/std mAP và lỗi vận hành; tự chọn theo mean mAP, không dùng test.
Không bảo đảm recipe giữ màu luôn thắng baseline; nếu không thắng sẽ giữ baseline.


In [ ]:
def per_class_metrics(metrics):
    return [{"class": metrics.names[int(class_id)],
             **dict(zip(("precision", "recall", "AP50", "AP50_95"),
                        map(float, metrics.class_result(index))))}
            for index, class_id in enumerate(metrics.box.ap_class_index)]


def train_stages(data_yaml, directory, recipe):
    common = dict(data=str(data_yaml), imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                  workers=0, seed=SEED, deterministic=True, optimizer="AdamW",
                  momentum=0.9, weight_decay=0.0005, cos_lr=True, lrf=0.1,
                  warmup_epochs=2.0, warmup_bias_lr=0.0, cache=False,
                  mosaic=0.0, mixup=0.0, copy_paste=0.0, cutmix=0.0,
                  hsv_h=0.0, hsv_s=0.0, hsv_v=0.05, degrees=5.0,
                  translate=0.03, scale=0.1, shear=0.0, perspective=0.0,
                  fliplr=0.5, flipud=0.0, project=str(directory), rect=False,
                  augmentations=RECIPES[recipe], save=True, plots=True, val=True,
                  resume=False, exist_ok=False)
    warmup = YOLO(PRETRAINED)
    warmup.train(**common, name="warmup", epochs=WARMUP_EPOCHS, patience=8, freeze=10, lr0=1e-3)
    warmup_best = Path(warmup.trainer.best)
    del warmup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    finetune = YOLO(str(warmup_best))
    finetune.train(**common, name="finetune", epochs=FINETUNE_EPOCHS, patience=15, freeze=0, lr0=1e-4)
    finetune_best = Path(finetune.trainer.best)
    del finetune
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    candidates = []
    for stage, checkpoint in (("warmup", warmup_best), ("finetune", finetune_best)):
        metrics = YOLO(str(checkpoint)).val(data=str(data_yaml), split="val", imgsz=IMGSZ,
                  rect=False, batch=BATCH, device=DEVICE, workers=0, conf=0.001, iou=0.7,
                  plots=True, project=str(directory), name=f"compare_{stage}", exist_ok=False)
        score = float(metrics.box.map)
        if not math.isfinite(score):
            raise ValueError(f"Validation score không hữu hạn: {stage}")
        candidates.append({"stage": stage, "checkpoint": str(checkpoint),
                           "val_mAP50_95": score, "classes": per_class_metrics(metrics)})
    selected = max(candidates, key=lambda candidate: candidate["val_mAP50_95"])
    best = directory / "best.pt"
    shutil.copy2(selected["checkpoint"], best)
    metadata = {"recipe": recipe, "model": PRETRAINED, "ultralytics": ultralytics.__version__,
                "torch": str(torch.__version__), "seed": SEED, "class_names": CLASS_NAMES,
                "imgsz": IMGSZ, "rect": False, "selected_stage": selected["stage"],
                "selection_metric": "validation mAP50-95", "candidates": candidates,
                "train_arguments": common, "warmup_epochs": WARMUP_EPOCHS,
                "finetune_epochs": FINETUNE_EPOCHS}
    (directory / "training_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    return best, selected


BEST_CHECKPOINT = RUN_DIR = TRAINING_DATA_YAML = None
if RUN_TRAINING and EXISTING_CHECKPOINT is not None:
    raise ValueError("Chọn train mới hoặc dùng EXISTING_CHECKPOINT, không bật đồng thời")
if RUN_TRAINING or EXISTING_CHECKPOINT is not None:
    if RUN_TRAINING and IS_KAGGLE and not torch.cuda.is_available():
        raise RuntimeError("Bật GPU Kaggle trước khi train")
    RUN_DIR = OUTPUT_ROOT / datetime.now().strftime("improved_%Y%m%d_%H%M%S_%f")
    RUN_DIR.mkdir(parents=True, exist_ok=False)
    (RUN_DIR / "audit.json").write_text(json.dumps(AUDIT, indent=2), encoding="utf-8")
    if RUN_TRAINING and RUN_CV:
        cv_rows = []
        for fold_index, fold in enumerate(FOLDS):
            fold_directory = RUN_DIR / f"fold_{fold_index + 1}"
            fold_yaml = stage_dataset(fold, fold_directory / "dataset")
            for recipe in RECIPES:
                directory = fold_directory / recipe
                checkpoint, selected = train_stages(fold_yaml, directory, recipe)
                operating = operating_evaluation(checkpoint, fold["valid"], directory / "operating")
                cv_rows.append({"fold": fold_index + 1, "recipe": recipe,
                                "mAP50_95": selected["val_mAP50_95"],
                                "damaged_to_riped_rate": operating["damaged_to_riped_rate"]})
                (RUN_DIR / "cv_results.json").write_text(json.dumps(cv_rows, indent=2), encoding="utf-8")
        cv_summary = []
        for recipe in RECIPES:
            rows = [row for row in cv_rows if row["recipe"] == recipe]
            scores = [row["mAP50_95"] for row in rows]
            cv_summary.append({"recipe": recipe, "mean_mAP50_95": statistics.mean(scores),
                               "std_mAP50_95": statistics.stdev(scores),
                               "mean_damaged_to_riped_rate": statistics.mean(row["damaged_to_riped_rate"] for row in rows)})
        SELECTED_RECIPE = max(cv_summary, key=lambda row: row["mean_mAP50_95"])["recipe"]
        (RUN_DIR / "cv_summary.json").write_text(json.dumps(cv_summary, indent=2), encoding="utf-8")
        print(json.dumps(cv_summary, indent=2), "Selected:", SELECTED_RECIPE)

    original_splits = {split: [r for r in RECORDS if r["split"] == split]
                       for split in ("train", "valid", "test")}
    TRAINING_DATA_YAML = stage_dataset(original_splits, RUN_DIR / "final" / "dataset")
    if RUN_TRAINING:
        BEST_CHECKPOINT, selected = train_stages(TRAINING_DATA_YAML, RUN_DIR / "final", SELECTED_RECIPE)
    else:
        source = Path(EXISTING_CHECKPOINT)
        if not source.is_file() or source.suffix != ".pt":
            raise ValueError("EXISTING_CHECKPOINT phải trỏ tới best.pt đã tải lên")
        if YOLO(str(source)).names != dict(enumerate(CLASS_NAMES)):
            raise ValueError("Class mapping checkpoint khác dataset")
        BEST_CHECKPOINT = RUN_DIR / "final" / "best.pt"
        shutil.copy2(source, BEST_CHECKPOINT)
    operating = operating_evaluation(BEST_CHECKPOINT, original_splits["valid"], RUN_DIR / "final" / "operating")
    print("Checkpoint:", BEST_CHECKPOINT, "| hư → chín:", operating["damaged_to_riped_rate"])
else:
    print("Chạy audit/preview, sau đó bật RUN_TRAINING hoặc điền EXISTING_CHECKPOINT để export.")


## 7. Test cuối: chủ động bật sau khi chốt model

Không dùng test để chọn recipe, checkpoint hoặc confidence. RUN_FINAL_TEST=False
mặc định. Khi bật, chạy riêng cell dưới sau train; không chạy lại toàn bộ notebook
vì sẽ tạo một lần train mới. Bảng P/R của val() vẫn là chỉ số tối ưu F1; báo cáo
operating riêng mới là kết quả đúng tại DEPLOY_CONF=0.4. Test cũ đã được xem để
phân tích lỗi: ghi rõ giới hạn này khi báo cáo. Test không có nền độc lập.


In [ ]:
if RUN_FINAL_TEST:
    if BEST_CHECKPOINT is None:
        raise RuntimeError("Cần checkpoint đã chốt")
    test_metrics = YOLO(str(BEST_CHECKPOINT)).val(
        data=str(TRAINING_DATA_YAML), split="test", imgsz=IMGSZ, rect=False,
        batch=BATCH, device=DEVICE, workers=0, conf=0.001, iou=0.7,
        plots=True, project=str(RUN_DIR), name="final_test", exist_ok=False)
    report = {"mAP50": float(test_metrics.box.map50), "mAP50_95": float(test_metrics.box.map),
              "classes": per_class_metrics(test_metrics)}
    (RUN_DIR / "test_metrics.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    operating_evaluation(BEST_CHECKPOINT, original_splits["test"], RUN_DIR / "final_test_operating")
    print(json.dumps(report, indent=2))
else:
    print("Test đang tắt.")


## 8. Export NCNN và so sánh với .pt

NCNN là một thư mục chứa .param, .bin và metadata.yaml, không chỉ một file đơn.
Export CPU, weights FP32 (quantize=32), input 640×640; giữ best.pt để fine-tune.
Ultralytics có thể tự cài ncnn/pnnx khi export; Kaggle cần Internet.

Hai backend được đánh giá trên cùng validation, CPU, batch=1, rect=False và 640.
Kiểm tra mAP giảm không quá 0,005 và số lỗi hư → chín không tăng ở conf=0.4.
Đây là cờ kiểm tra, không phải bảo đảm model đạt chuẩn. Nếu không đạt, xem báo cáo
trước khi triển khai. Tốc độ CPU Kaggle không thay thế benchmark trên Pi.

Pipeline Pi hiện chỉ nhận .pt: ZIP NCNN chưa thể dùng chỉ bằng đổi đường dẫn TOML.
Cần thay loader/config trong một task riêng trước khi chạy băng tải bằng NCNN.


In [ ]:
if RUN_EXPORT_NCNN and BEST_CHECKPOINT is not None:
    ncnn_directory = Path(YOLO(str(BEST_CHECKPOINT)).export(
        format="ncnn", imgsz=IMGSZ, device="cpu", quantize=32))
    if not (list(ncnn_directory.glob("*.param")) and list(ncnn_directory.glob("*.bin"))
            and (ncnn_directory / "metadata.yaml").is_file()):
        raise RuntimeError("Export NCNN thiếu param/bin/metadata")
    validation_rows = original_splits["valid"]
    backend_reports = {}
    for backend, checkpoint in (("pytorch", BEST_CHECKPOINT), ("ncnn", ncnn_directory)):
        metrics = YOLO(str(checkpoint), task="detect").val(
            data=str(TRAINING_DATA_YAML), split="val", imgsz=IMGSZ, rect=False,
            batch=1, device="cpu", workers=0, conf=0.001, iou=0.7,
            plots=True, project=str(RUN_DIR), name=f"export_compare_{backend}", exist_ok=False)
        if metrics.names != dict(enumerate(CLASS_NAMES)):
            raise ValueError(f"Class mapping backend sai: {backend}")
        operating = operating_evaluation(checkpoint, validation_rows,
                                         RUN_DIR / f"export_operating_{backend}", device="cpu")
        backend_reports[backend] = {"mAP50_95": float(metrics.box.map),
                                    "classes": per_class_metrics(metrics),
                                    "damaged_to_riped": operating["damaged_to_riped"]}
    drop = backend_reports["pytorch"]["mAP50_95"] - backend_reports["ncnn"]["mAP50_95"]
    passed = (math.isfinite(drop) and drop <= NCNN_MAX_MAP_DROP
              and backend_reports["ncnn"]["damaged_to_riped"] <= backend_reports["pytorch"]["damaged_to_riped"])
    export_report = {"backends": backend_reports, "mAP_drop": drop,
                     "max_allowed_mAP_drop": NCNN_MAX_MAP_DROP, "validation_check_passed": passed,
                     "imgsz": IMGSZ, "rect": False, "confidence": DEPLOY_CONF,
                     "quantize": 32, "ultralytics": ultralytics.__version__}
    (RUN_DIR / "ncnn_validation.json").write_text(json.dumps(export_report, indent=2), encoding="utf-8")
    package = RUN_DIR / "deployment"
    package.mkdir(exist_ok=False)
    shutil.copytree(ncnn_directory, package / ncnn_directory.name)
    shutil.copy2(RUN_DIR / "ncnn_validation.json", package / "ncnn_validation.json")
    (package / "README.txt").write_text(
        "Load the complete *_ncnn_model directory with Ultralytics YOLO, task=detect.\n"
        "Predict on CPU with imgsz=640, rect=False, conf=0.4; keep metadata.yaml.\n"
        "Check ncnn_validation.json and benchmark on Pi before conveyor use.\n"
        "The current conveyor loader accepts .pt only; integration is a separate change.\n",
        encoding="utf-8")
    archive = shutil.make_archive(str(RUN_DIR / "ce224_ncnn"), "zip", root_dir=package)
    print("NCNN:", ncnn_directory, "| ZIP:", archive)
    print("Validation check:", "PASS" if passed else "CẦN KIỂM TRA, chưa nên triển khai")
elif RUN_EXPORT_NCNN:
    print("Chưa có weights để export: train mới hoặc điền EXISTING_CHECKPOINT.")


## 9. Tải kết quả và bước tiếp theo

Kaggle Output → ce224_training/improved_<timestamp>:
- cv_results.json, cv_summary.json: so sánh cùng fold, mean/std, lỗi hư → chín.
- final/best.pt, final/training_metadata.json: model cuối và cấu hình (train mới).
- final/operating/: ảnh lỗi và số liệu từng quả tại confidence vận hành.
- ce224_ncnn.zip, ncnn_validation.json: model NCNN đầy đủ và kiểm tra chuyển đổi.
- test_metrics.json/final_test_operating chỉ có khi bật test.

Save Version với output hoặc tải ZIP/weights trước khi đóng phiên. Recipe thắng
CV có thể vẫn là baseline: không tuyên bố cải thiện nếu kết quả không ủng hộ.
Nếu lỗi tập trung vào quả vàng-hư, kiểm tra các nhóm train và tiêu chí nhãn;
không lấy frame của quả test vào train. Có thể thử imgsz=832 ở một thí nghiệm
riêng sau này, nhưng cần đo chi phí trên Pi và so sánh lại cùng folds.
